In [ ]:
from google import genai
from google.genai import types
import json
import time
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# API key
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    raise ValueError("Please set GEMINI_API_KEY in .env file")

client = genai.Client(api_key=API_KEY)

# File paths
INPUT_FILE = Path("../data/final_quiz_data.json")
OUTPUT_FILE = Path("../data/final_quiz_data_translated.json")  # Will create new  file
PROGRESS_FILE = Path("../data/translation_progress.json")

print("✓ Setup complete")

✓ Setup complete


## Translation Functions

In [28]:
def translate_to_french(arabic_text, context=""):
    """Translate Arabic text to French using Gemini API"""
    try:
        prompt = f"""Translate this Arabic driving test answer to French.
                    Context: {context}

                    Arabic text: {arabic_text}
                    
                    Return ONLY the French translation, no explanations or extra text."""
        
        response = client.models.generate_content(
            model="gemini-2.0-flash",
            contents=prompt
        )
        
        return response.text.strip(), None
    
    except Exception as e:
        return None, str(e)


def load_progress():
    """Load translation progress"""
    if PROGRESS_FILE.exists():
        with open(PROGRESS_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_progress(progress):
    """Save translation progress"""
    with open(PROGRESS_FILE, 'w', encoding='utf-8') as f:
        json.dump(progress, f, ensure_ascii=False, indent=2)


def save_quiz_data(data):
    """Save updated quiz data immediately"""
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

print("✓ Functions ready")

✓ Functions ready


## Load Quiz Data

In [5]:
# Load quiz data
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    quiz_data = json.load(f)

# Load progress
progress = load_progress()

print(f"Loaded {len(quiz_data)} tests")
print(f"Translation progress: {len(progress)} items completed")

Loaded 20 tests
Translation progress: 0 items completed


## Count Total Items to Translate

In [10]:
# Count items
total_signs = 0
total_priorities = 0
total_questions = 0

for test_name, test_data in quiz_data.items():
    sections = test_data.get('sections', {})
    total_signs += len(sections.get('road_signs', []))
    total_priorities += len(sections.get('priorities', []))
    total_questions += len(sections.get('general_questions', []))

total_items = total_signs + total_priorities + total_questions

print(f"Total items to translate:")
print(f"  Road Signs: {total_signs}")
print(f"  Priorities: {total_priorities}")
print(f"  General Questions: {total_questions}")
print(f"  TOTAL: {total_items}")

Total items to translate:
  Road Signs: 320
  Priorities: 160
  General Questions: 119
  TOTAL: 599


## Translate Road Signs Answers

In [2]:
print("Translating Road Signs...\n")
signs_translated = 0
signs_skipped = 0
signs_failed = 0

for test_name in sorted(quiz_data.keys()):
    test_data = quiz_data[test_name]
    road_signs = test_data.get('sections', {}).get('road_signs', [])
    
    for i, sign in enumerate(road_signs):
        # Create unique ID for tracking
        item_id = f"{test_name}_sign_{sign['sign_number']}"
        
        # Skip if already translated
        if 'answer_french' in sign and sign['answer_french']:
            signs_skipped += 1
            continue
        
        # Skip if in progress cache
        if item_id in progress:
            sign['answer_arabic'] = sign.pop('correct_answer', sign.get('answer_arabic', ''))
            sign['answer_french'] = progress[item_id]
            signs_skipped += 1
            continue
        
        # Translate
        arabic_answer = sign.get('correct_answer', sign.get('answer_arabic', ''))
        
        print(f"→ {test_name} Sign {sign['sign_number']}: Translating...", end=" ", flush=True)
        
        french_answer, error = translate_to_french(arabic_answer, "Road sign answer")
        
        if error:
            print(f"✗ Error: {error}")
            signs_failed += 1
            time.sleep(10)
            continue
        
        # Update data structure
        sign['answer_arabic'] = arabic_answer
        sign['answer_french'] = french_answer
        if 'correct_answer' in sign:
            del sign['correct_answer']
        
        # Save immediately
        progress[item_id] = french_answer
        save_progress(progress)
        save_quiz_data(quiz_data)
        
        signs_translated += 1
        print("✓")
        time.sleep(10)  # Rate limit

print(f"\n{'='*60}")
print(f"Road Signs: {signs_translated} translated, {signs_skipped} skipped, {signs_failed} failed")
print(f"{'='*60}")

Translating Road Signs...

→ test-01 Sign 1: Translating... ✗ Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. ', 'status': 'RESOURCE_EXHAUSTED'}}
✗ Error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. ', 'status': 'RESOURCE_EXHAUSTED'}}


KeyboardInterrupt: 

## Translate Priorities Answers

In [11]:
print("Translating Priorities...\n")
priorities_translated = 0
priorities_skipped = 0
priorities_failed = 0

for test_name in sorted(quiz_data.keys()):
    test_data = quiz_data[test_name]
    priorities = test_data.get('sections', {}).get('priorities', [])
    
    for i, priority in enumerate(priorities):
        # Create unique ID
        item_id = f"{test_name}_priority_{priority['priority_number']}"
        
        # Skip if already translated
        if 'answer_french' in priority and priority['answer_french']:
            priorities_skipped += 1
            continue
        
        # Skip if in progress cache
        if item_id in progress:
            priority['answer_arabic'] = priority.pop('correct_answer', priority.get('answer_arabic', ''))
            priority['answer_french'] = progress[item_id]
            priorities_skipped += 1
            continue
        
        # Translate
        arabic_answer = priority.get('correct_answer', priority.get('answer_arabic', ''))
        
        print(f"→ {test_name} Priority {priority['priority_number']}: Translating...", end=" ", flush=True)
        
        french_answer, error = translate_to_french(arabic_answer, "Priority traffic situation answer")
        
        if error:
            print(f"✗ Error: {error}")
            priorities_failed += 1
            time.sleep(5)
            continue
        
        # Update data structure
        priority['answer_arabic'] = arabic_answer
        priority['answer_french'] = french_answer
        if 'correct_answer' in priority:
            del priority['correct_answer']
        
        # Save immediately
        progress[item_id] = french_answer
        save_progress(progress)
        save_quiz_data(quiz_data)
        
        priorities_translated += 1
        print("✓")
        time.sleep(5)  # Rate limit

print(f"\n{'='*60}")
print(f"Priorities: {priorities_translated} translated, {priorities_skipped} skipped, {priorities_failed} failed")
print(f"{'='*60}")

Translating Priorities...

→ test-01 Priority 1: Translating... ✗ Error: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}
✗ Error: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}
→ test-01 Priority 2: Translating... → test-01 Priority 2: Translating... ✗ Error: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}
✗ Error: 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your API key was reported as leaked. Please use another API key.', 'status': 'PERMISSION_DENIED'}}


KeyboardInterrupt: 

## Translate General Questions Answers

In [ ]:
print("Translating General Questions...\n")
questions_translated = 0
questions_skipped = 0
questions_failed = 0

for test_name in sorted(quiz_data.keys()):
    test_data = quiz_data[test_name]
    questions = test_data.get('sections', {}).get('general_questions', [])
    
    for i, question in enumerate(questions):
        # Create unique ID
        item_id = f"{test_name}_question_{question['question_number']}"
        
        # Skip if already translated
        if 'answer_french' in question and question['answer_french']:
            questions_skipped += 1
            continue
        
        # Skip if in progress cache
        if item_id in progress:
            question['answer_arabic'] = question.pop('correct_answer', question.get('answer_arabic', ''))
            question['answer_french'] = progress[item_id]
            questions_skipped += 1
            continue
        
        # Translate
        arabic_answer = question.get('correct_answer', question.get('answer_arabic', ''))
        
        print(f"→ {test_name} Question {question['question_number']}: Translating...", end=" ", flush=True)
        
        # Use French question as context for better translation
        context = f"Question: {question.get('question_fr', '')}"
        french_answer, error = translate_to_french(arabic_answer, context)
        
        if error:
            print(f"✗ Error: {error}")
            questions_failed += 1
            time.sleep(5)
            continue
        
        # Update data structure
        question['answer_arabic'] = arabic_answer
        question['answer_french'] = french_answer
        if 'correct_answer' in question:
            del question['correct_answer']
        
        # Save immediately
        progress[item_id] = french_answer
        save_progress(progress)
        save_quiz_data(quiz_data)
        
        questions_translated += 1
        print("✓")
        time.sleep(5)  # Rate limit

print(f"\n{'='*60}")
print(f"General Questions: {questions_translated} translated, {questions_skipped} skipped, {questions_failed} failed")
print(f"{'='*60}")

## Final Summary & Verification

In [ ]:
# Reload data to verify
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    final_data = json.load(f)

# Count completeness
total_complete = 0
total_incomplete = 0

for test_name, test_data in final_data.items():
    sections = test_data.get('sections', {})
    
    for sign in sections.get('road_signs', []):
        if 'answer_french' in sign and sign['answer_french']:
            total_complete += 1
        else:
            total_incomplete += 1
    
    for priority in sections.get('priorities', []):
        if 'answer_french' in priority and priority['answer_french']:
            total_complete += 1
        else:
            total_incomplete += 1
    
    for question in sections.get('general_questions', []):
        if 'answer_french' in question and question['answer_french']:
            total_complete += 1
        else:
            total_incomplete += 1

print(f"\n{'='*60}")
print(f"FINAL SUMMARY")
print(f"{'='*60}")
print(f"Total items: {total_items}")
print(f"Completed: {total_complete} ({total_complete/total_items*100:.1f}%)")
print(f"Incomplete: {total_incomplete}")
print(f"\nTotal translated this session:")
print(f"  Road Signs: {signs_translated}")
print(f"  Priorities: {priorities_translated}")
print(f"  Questions: {questions_translated}")
print(f"  TOTAL: {signs_translated + priorities_translated + questions_translated}")
print(f"\nTotal failed:")
print(f"  Road Signs: {signs_failed}")
print(f"  Priorities: {priorities_failed}")
print(f"  Questions: {questions_failed}")
print(f"  TOTAL: {signs_failed + priorities_failed + questions_failed}")
print(f"{'='*60}")

if total_incomplete > 0:
    print(f"\n⚠️  Run this notebook again to translate remaining {total_incomplete} items")
else:
    print(f"\n✓ ALL TRANSLATIONS COMPLETE!")

## View Sample Translations

In [ ]:
# Show sample from first test
first_test = final_data['test-01']
sections = first_test['sections']

print("Sample Road Sign:")
if sections['road_signs']:
    sign = sections['road_signs'][0]
    print(f"Sign #{sign['sign_number']}")
    print(f"Arabic: {sign.get('answer_arabic', 'N/A')[:80]}...")
    print(f"French: {sign.get('answer_french', 'N/A')[:80]}...")

print("\nSample Priority:")
if sections['priorities']:
    priority = sections['priorities'][0]
    print(f"Priority #{priority['priority_number']}")
    print(f"Arabic: {priority.get('answer_arabic', 'N/A')[:80]}...")
    print(f"French: {priority.get('answer_french', 'N/A')[:80]}...")

print("\nSample General Question:")
if sections['general_questions']:
    q = sections['general_questions'][0]
    print(f"Question #{q['question_number']}")
    print(f"Question FR: {q.get('question_fr', 'N/A')[:80]}...")
    print(f"Answer AR: {q.get('answer_arabic', 'N/A')[:80]}...")
    print(f"Answer FR: {q.get('answer_french', 'N/A')[:80]}...")